# NB06：進階索引策略與 Context 管理

**目標：** 掌握讓 RAG 在真實世界長文件場景下穩定運作的核心技術。

**學習成果：**
- 理解並實作 Parent-Child Chunking（小塊檢索，大塊合成）
- 掌握 Sentence-Window Retrieval（句子窗口檢索）
- 學會 Contextual Compression 節省 token 成本
- 了解「Lost in the Middle」問題及緩解策略
- 實作 Map-Reduce 與 Refine 兩種長文件問答模式

---

## 問題背景：Naive Chunking 的兩大痛點

NB02 介紹了基本切塊策略，但在真實場景中會遇到根本性矛盾：

| 切塊大小 | 向量搜索效果 | LLM Context 品質 |
|---------|------------|----------------|
| **小塊（100 tokens）** | ✅ 精準，語義集中 | ❌ 缺乏上下文，LLM 答案片段化 |
| **大塊（600 tokens）** | ❌ 雜訊多，語義模糊 | ✅ 上下文豐富，LLM 回答完整 |

**核心矛盾：** 向量搜索喜歡小塊；LLM 需要大塊。本 Notebook 的技術都是在解決這個矛盾。

### 本 Notebook 涵蓋的技術

```
進階索引策略
├── Part 1: Parent-Child Chunking  → 小塊搜索 + 大塊合成
├── Part 2: Sentence-Window        → 句子精準匹配 + 窗口補充上下文
├── Part 3: Contextual Compression → 壓縮 chunk，過濾無關內容
├── Part 4: Lost in the Middle     → Context 位置效應與緩解策略
└── Part 5: Map-Reduce / Refine    → 超長文件問答架構
```

## 環境設定

In [ ]:
import os
import re
import time
import textwrap
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
import tiktoken
import matplotlib.pyplot as plt
import matplotlib

# 設定中文字型（Mac 環境）
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'PingFang TC', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

# 載入 .env 的 API Key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ChromaDB 客戶端
chroma_client = chromadb.Client()

# tiktoken 計算 token 數
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """計算文字的 token 數量"""
    return len(enc.encode(text))

def get_embedding(text: str) -> List[float]:
    """使用 OpenAI text-embedding-3-small 生成嵌入向量"""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def ask_llm(prompt: str, system: str = "你是一個專業的繁體中文助理。", temperature: float = 0.1) -> str:
    """呼叫 gpt-4o-mini 取得回答"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

print("✅ 環境設定完成")
print(f"OpenAI API Key 已載入: {bool(os.getenv('OPENAI_API_KEY'))}")

## 示範資料：LLM 架構技術文章

以下是一篇約 2000 字的技術文章，作為本 Notebook 所有示範的資料來源。

In [ ]:
# 示範文件：約 2000 字的 LLM 架構技術文章
DEMO_ARTICLE = """
大型語言模型（LLM）架構深度解析

一、Transformer 架構基礎

Transformer 模型由 Vaswani 等人於 2017 年在論文「Attention Is All You Need」中提出。
它徹底改變了自然語言處理領域，成為現代大型語言模型的基石。
相比於 RNN 和 LSTM，Transformer 最大的優勢在於能夠並行處理序列資料，
大幅縮短了訓練時間。其核心機制是自注意力（Self-Attention），
能夠讓模型在處理每個詞時，同時考慮序列中所有其他詞的資訊。

二、自注意力機制（Self-Attention）

自注意力機制的核心是三個矩陣：Query（Q）、Key（K）和 Value（V）。
對於輸入序列中的每個詞，模型會計算它與所有其他詞之間的相關性分數。
計算公式為：Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V。
其中 d_k 是 Key 向量的維度，除以 sqrt(d_k) 是為了防止梯度消失問題。
多頭注意力（Multi-Head Attention）是自注意力的擴展，
同時執行多個注意力計算，讓模型能從不同角度理解文本關係。
GPT-4 使用了 96 個注意力頭，每個頭的維度為 128。

三、預訓練方法

現代 LLM 主要使用兩種預訓練目標：

（一）掩碼語言模型（Masked Language Model，MLM）：
由 BERT 所採用，隨機遮住輸入序列中 15% 的詞彙，
訓練模型根據上下文預測被遮住的詞。
這種方法讓模型能夠學習雙向的文本理解能力，
特別適合文本分類、實體識別等理解任務。

（二）因果語言模型（Causal Language Model，CLM）：
由 GPT 系列採用，訓練模型預測序列中下一個詞。
模型只能看到當前位置之前的詞（單向），
這種方式天然適合文本生成任務。
GPT-4、Claude、Gemini 等主流助理模型都使用此方法。

四、規模定律（Scaling Laws）

Kaplan 等人在 2020 年的研究發現，LLM 的性能與三個因素呈現冪律關係：
模型參數量（N）、訓練資料量（D）和計算量（C）。
具體而言，在計算預算固定的情況下，
增加模型大小和增加訓練資料量的效益相當。
DeepMind 在 2022 年發布的 Chinchilla 研究進一步修正了這個觀點，
指出許多大模型其實是「計算量最優」但「訓練資料不足」的狀態。
Chinchilla 定律建議：模型參數量每增加一倍，訓練 token 數也應增加一倍。
以 Llama 2 70B 為例，它使用了 2 兆個 token 進行訓練。

五、位置編碼（Positional Encoding）

Transformer 本身不含序列位置資訊，需要額外加入位置編碼。
原始 Transformer 使用正弦/餘弦函數生成固定的位置向量。
現代 LLM 多採用 RoPE（Rotary Position Embedding，旋轉位置嵌入），
由 Su 等人提出，能夠更好地外推至訓練時未見過的更長序列。
LLaMA、Mistral、ChatGLM 等開源模型都採用 RoPE。
ALiBi（Attention with Linear Biases）是另一種方案，
直接在注意力分數上加入線性偏置，實作更簡單。

六、Context Window 的限制與擴展

Context window（上下文窗口）是 LLM 一次能處理的最大 token 數量。
早期的 GPT-3 只有 4096 tokens，而 Gemini 1.5 Pro 已達到 100 萬 tokens。
增大 context window 面臨的挑戰包括：
（1）計算複雜度：自注意力的計算複雜度是序列長度的平方 O(n²)，
     序列長一倍，計算量增加四倍。
（2）記憶體消耗：KV Cache 需要儲存所有層的 Key 和 Value 矩陣。
（3）位置外推：模型在訓練長度之外的泛化能力較差。
Flash Attention 技術通過重新組織計算順序，
將 attention 的記憶體複雜度從 O(n²) 降至 O(n)，
是目前主流的效率優化方案。

七、指令微調與 RLHF

預訓練完成後，LLM 還需要進行「對齊」（Alignment）訓練，
讓它學會遵循人類指令並產生有幫助的回答。

指令微調（Instruction Fine-tuning，IFT）：
使用大量「指令-回應」對進行監督式學習，
讓模型學習如何按照人類指令回應。
Stanford Alpaca 使用 GPT-4 生成 52,000 筆指令資料，以 LLaMA 為底座微調。

人類反饋強化學習（RLHF）：
首先訓練一個獎勵模型（Reward Model），學習人類偏好。
然後使用 PPO（Proximal Policy Optimization）演算法，
以獎勵模型的分數作為信號，優化 LLM 的生成策略。
InstructGPT 和 ChatGPT 都使用了 RLHF 訓練。

DPO（Direct Preference Optimization）是 RLHF 的簡化版，
不需要獨立的獎勵模型，直接在偏好資料上優化 LLM，
訓練更穩定、成本更低，近期被廣泛採用。

八、推理效率優化

生產環境中，LLM 推理效率至關重要。主要優化技術包括：

量化（Quantization）：將模型權重從 FP32 或 FP16 壓縮到 INT8 或 INT4，
大幅降低記憶體需求和推理延遲。GPTQ 和 AWQ 是目前主流的量化方案。

KV Cache：快取已計算過的 Key 和 Value 矩陣，
避免在生成每個新 token 時重複計算。
這是 LLM 推理加速的最基本也最重要的技術。

投機解碼（Speculative Decoding）：使用一個小模型快速生成草稿，
再由大模型並行驗證，可將吞吐量提升 2-3 倍。

批次推理（Batching）：連續批次處理（Continuous Batching）
允許在生成過程中動態加入新請求，大幅提升 GPU 利用率。
vLLM 使用 PagedAttention 技術實現高效批次推理，
是目前最主流的 LLM 推理框架之一。
"""

token_count = count_tokens(DEMO_ARTICLE)
print(f"示範文章長度：{len(DEMO_ARTICLE)} 字元，約 {token_count} tokens")
print(f"文章章節：")
sections = [line.strip() for line in DEMO_ARTICLE.split('\n') if line.strip().startswith(('一、', '二、', '三、', '四、', '五、', '六、', '七、', '八、'))]
for s in sections:
    print(f"  {s}")

---

## Part 1：Parent-Child Chunking（小塊檢索，大塊合成）

### 核心概念

Parent-Child Chunking 的精髓在於**分離「搜索單元」與「合成單元」**：

```
文件
├── Parent Chunk 1 (500 tokens) ← 送給 LLM 的 context（豐富）
│   ├── Child Chunk 1a (100 tokens) ← 用於向量搜索（精準）
│   ├── Child Chunk 1b (100 tokens) ← 用於向量搜索
│   └── Child Chunk 1c (100 tokens) ← 用於向量搜索
└── Parent Chunk 2 (500 tokens)
    ├── Child Chunk 2a (100 tokens)
    └── ...
```

**為什麼有效？**
- 小 child chunk → embedding 語義集中，搜索更精準
- 匹配到 child → 取回其 parent chunk → LLM 得到充足上下文
- 多個 child 指向同一 parent 時，自動去重（避免重複送同一段文字）

**適用場景：** 技術文件、百科知識庫、FAQ 問答。

In [ ]:
# ── 通用切塊輔助函式 ──────────────────────────────────

def split_text_by_tokens(text: str, chunk_size: int, overlap: int = 0) -> List[str]:
    """
    依照 token 數切塊。
    chunk_size: 每塊的最大 token 數
    overlap:    相鄰 chunk 的重疊 token 數
    """
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        if end == len(tokens):
            break
        start += chunk_size - overlap
    return chunks

# 測試切塊
test_chunks = split_text_by_tokens(DEMO_ARTICLE, chunk_size=200, overlap=20)
print(f"文章切成 {len(test_chunks)} 塊（chunk_size=200, overlap=20）")
print(f"第 1 塊 token 數：{count_tokens(test_chunks[0])}")
print(f"第 2 塊 token 數：{count_tokens(test_chunks[1])}")

In [ ]:
# ── Part 1：建立 Parent-Child 索引 ────────────────────

def create_parent_child_index(
    text: str,
    child_size: int = 150,
    parent_size: int = 600,
    overlap: int = 30,
    collection_name: str = "parent_child_demo"
) -> Tuple[object, Dict[str, str]]:
    """
    建立 Parent-Child 兩層索引。

    返回：
        child_collection: 存放小塊向量的 ChromaDB collection
        parent_store:     dict，parent_id → parent_text
    """
    # 先切 parent chunks（大塊）
    parent_chunks = split_text_by_tokens(text, chunk_size=parent_size, overlap=overlap)

    # 建立 ChromaDB collection 存 child 向量
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    child_collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    parent_store: Dict[str, str] = {}  # parent_id → parent_text
    child_id_counter = 0

    for p_idx, parent_text in enumerate(parent_chunks):
        parent_id = f"parent_{p_idx}"
        parent_store[parent_id] = parent_text

        # 將 parent 再切成 child chunks（小塊）
        child_chunks = split_text_by_tokens(parent_text, chunk_size=child_size, overlap=0)

        for c_idx, child_text in enumerate(child_chunks):
            child_id = f"child_{child_id_counter}"
            child_id_counter += 1

            embedding = get_embedding(child_text)
            child_collection.add(
                ids=[child_id],
                embeddings=[embedding],
                documents=[child_text],
                metadatas=[{"parent_id": parent_id, "child_index": c_idx}]
            )

    print(f"索引建立完成：")
    print(f"  Parent chunks: {len(parent_chunks)} 塊（≈{parent_size} tokens/塊）")
    print(f"  Child  chunks: {child_id_counter} 塊（≈{child_size} tokens/塊）")
    return child_collection, parent_store


# 建立索引
pc_collection, pc_parent_store = create_parent_child_index(
    DEMO_ARTICLE,
    child_size=150,
    parent_size=600,
    overlap=30,
    collection_name="nb06_parent_child"
)

In [ ]:
# ── Part 1：Parent-Child 檢索函式 ────────────────────

def parent_child_retrieve(
    query: str,
    child_collection: object,
    parent_store: Dict[str, str],
    n_results: int = 3
) -> List[str]:
    """
    1. 搜索 child collection（小塊，精準匹配）
    2. 取回每個 child 對應的 parent（大塊，豐富 context）
    3. 去重（多個 child 可能指向同一 parent）
    """
    query_embedding = get_embedding(query)
    results = child_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results * 2,  # 多取一些 child，防止 parent 去重後不足
        include=["documents", "metadatas", "distances"]
    )

    seen_parent_ids = set()
    parent_contexts = []

    for child_text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        parent_id = meta["parent_id"]
        if parent_id not in seen_parent_ids:
            seen_parent_ids.add(parent_id)
            parent_contexts.append({
                "parent_id": parent_id,
                "parent_text": parent_store[parent_id],
                "matched_child": child_text,
                "similarity": round(1 - dist, 4)
            })
        if len(parent_contexts) >= n_results:
            break

    return parent_contexts


# ── 對比：標準切塊檢索 ────────────────────────────────

def build_standard_index(text: str, chunk_size: int = 300, collection_name: str = "nb06_standard") -> object:
    """建立普通 flat 索引（對比用）"""
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    chunks = split_text_by_tokens(text, chunk_size=chunk_size, overlap=30)
    for i, chunk in enumerate(chunks):
        collection.add(
            ids=[f"chunk_{i}"],
            embeddings=[get_embedding(chunk)],
            documents=[chunk]
        )
    print(f"標準索引建立完成：{len(chunks)} 塊（≈{chunk_size} tokens/塊）")
    return collection


std_collection = build_standard_index(DEMO_ARTICLE, chunk_size=300, collection_name="nb06_standard")

In [ ]:
# ── Part 1：對比示範 ──────────────────────────────────

query = "MLM 訓練方法隨機遮住多少比例的詞彙？"
print(f"查詢問題：{query}")
print("=" * 70)

# 方法 A：標準切塊（chunk_size=300）
query_emb = get_embedding(query)
std_results = std_collection.query(
    query_embeddings=[query_emb],
    n_results=1,
    include=["documents", "distances"]
)
std_chunk = std_results["documents"][0][0]
std_sim = 1 - std_results["distances"][0][0]

print("\n【方法 A：標準切塊（chunk_size=300）】")
print(f"相似度：{std_sim:.4f}")
print(f"取回 token 數：{count_tokens(std_chunk)}")
print(f"內容（前 200 字）：\n{std_chunk[:200]}...")

print("\n" + "─" * 70)

# 方法 B：Parent-Child（child=150，parent=600）
pc_results = parent_child_retrieve(query, pc_collection, pc_parent_store, n_results=1)
pc_result = pc_results[0]

print("\n【方法 B：Parent-Child（child=150, parent=600）】")
print(f"匹配到的 Child 相似度：{pc_result['similarity']:.4f}")
print(f"匹配 Child 內容：\n  {pc_result['matched_child'][:120]}...")
print(f"\n取回 Parent token 數：{count_tokens(pc_result['parent_text'])}")
print(f"Parent 內容（前 300 字）：\n{pc_result['parent_text'][:300]}...")

### Part 1 分析

**觀察重點：**
- Child chunk 精準定位到包含「15%」的那一小段，相似度高
- 回傳給 LLM 的卻是 Parent（600 tokens），包含了整個「預訓練方法」章節
- LLM 不只能回答「15%」，還能進一步解釋 MLM vs CLM 的差異

**標準切塊的問題：** 如果問題的答案恰好位於 chunk 邊界，可能被截斷；
Parent-Child 的 parent 邊界也可能截斷，但機率大幅降低，因為 parent 較大。

**成本考量：** Parent-Child 需要建立更多嵌入向量（child 數量 >> parent 數量），
索引成本較高，但檢索品質顯著提升，在生產環境中通常值得。

---

## Part 2：Sentence-Window Retrieval（句子窗口檢索）

### 核心概念

Sentence-Window 是 Parent-Child 的「極端版本」：
- **搜索單元 = 單一句子**（最小語義單位）
- **合成單元 = 句子 ± N 個鄰居句子**（提供上下文窗口）

```
原始文本句子序列：
[句0] [句1] [句2] [→ 句3 匹配 ←] [句4] [句5] [句6]

window_size=2 時，送給 LLM 的 context：
[句1] [句2] [句3] [句4] [句5]  （匹配句 ±2）
```

**最佳適用場景：**
- 精確事實問答（答案是一個特定的句子）
- 需要知道某個事實的「前因後果」
- 文件中有大量不相關章節的「針尖問題」

In [ ]:
# ── Part 2：句子切分輔助函式 ─────────────────────────

def split_into_sentences(text: str) -> List[str]:
    """
    將文字切分成句子。
    針對中文文本，以句號、問號、驚嘆號、換行等作為分隔符。
    """
    # 先按換行分，再按標點分
    text = text.strip()
    # 以中英文句末標點作為斷點（保留標點在句子末尾）
    raw = re.split(r'(?<=[。！？.!?\n])\s*', text)
    sentences = [s.strip() for s in raw if s.strip() and len(s.strip()) > 5]
    return sentences


def build_sentence_window_index(
    text: str,
    collection_name: str = "nb06_sentence_window"
) -> Tuple[object, List[str]]:
    """
    建立句子級別索引。
    返回：
        sentence_collection: ChromaDB collection（每個文件 = 一個句子）
        sentences:           List[str]，原始句子陣列（用於取回窗口）
    """
    sentences = split_into_sentences(text)

    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    sentence_collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    for idx, sentence in enumerate(sentences):
        sentence_collection.add(
            ids=[f"sent_{idx}"],
            embeddings=[get_embedding(sentence)],
            documents=[sentence],
            metadatas=[{"index": idx}]
        )

    print(f"句子索引建立完成：{len(sentences)} 個句子")
    print(f"平均每句 token 數：{sum(count_tokens(s) for s in sentences) / len(sentences):.1f}")
    return sentence_collection, sentences


sw_collection, sw_sentences = build_sentence_window_index(
    DEMO_ARTICLE,
    collection_name="nb06_sentence_window"
)

In [ ]:
# ── Part 2：句子窗口檢索函式 ──────────────────────────

def sentence_window_retrieve(
    query: str,
    sentence_collection: object,
    sentences: List[str],
    n_results: int = 2,
    window_size: int = 2
) -> List[Dict]:
    """
    1. 找出語意最相符的句子
    2. 每個匹配句子 → 取 [idx-window_size, idx+window_size] 範圍的句子作為 context
    """
    query_embedding = get_embedding(query)
    results = sentence_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    contexts = []
    for sent_text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        idx = meta["index"]
        # 取窗口範圍（不超出邊界）
        start = max(0, idx - window_size)
        end = min(len(sentences), idx + window_size + 1)
        window_text = " ".join(sentences[start:end])

        contexts.append({
            "matched_sentence": sent_text,
            "sentence_index": idx,
            "window_text": window_text,
            "window_tokens": count_tokens(window_text),
            "similarity": round(1 - dist, 4)
        })

    return contexts


# ── 示範 ─────────────────────────────────────────────
query = "Flash Attention 如何降低記憶體複雜度？"
print(f"查詢問題：{query}")
print("=" * 70)

sw_results = sentence_window_retrieve(
    query, sw_collection, sw_sentences, n_results=2, window_size=2
)

for i, res in enumerate(sw_results):
    print(f"\n【第 {i+1} 個匹配結果】")
    print(f"精準匹配句子（第 {res['sentence_index']} 句）：")
    print(f"  → {res['matched_sentence']}")
    print(f"相似度：{res['similarity']:.4f}")
    print(f"\n送給 LLM 的窗口 context（window_size=2，{res['window_tokens']} tokens）：")
    print(textwrap.fill(res['window_text'], width=80, initial_indent='  ', subsequent_indent='  '))

### Part 2 分析

**Sentence-Window 的優缺點比較：**

| 面向 | Sentence-Window | Parent-Child |
|------|----------------|--------------|
| 搜索精度 | ⭐⭐⭐⭐⭐（最高） | ⭐⭐⭐⭐ |
| Context 完整性 | ⭐⭐⭐（窗口可能截斷段落） | ⭐⭐⭐⭐⭐ |
| 索引成本 | ⭐⭐（每句都要 embed） | ⭐⭐⭐（child 數量適中） |
| 實作複雜度 | ⭐⭐⭐（需要句子切分） | ⭐⭐⭐⭐（需要兩層切塊） |

**實務建議：** window_size 通常設 2-3。太小則 context 不足，太大則等同於直接用大塊，失去精準搜索的意義。

---

## Part 3：Contextual Compression（上下文壓縮）

### 核心概念

向量搜索取回的 chunk 通常「什麼都有一點」。對於一個具體問題，
chunk 中大部分內容可能是無關的「雜訊」：

```
檢索到的 Chunk（500 tokens）：
"...Transformer 由 Vaswani 提出...（不相關）
 ...MLM 隨機遮住 15% 的詞彙，訓練模型預測...（相關！）
 ...GPT 系列採用 CLM...（不相關）..."
                 ↓ Contextual Compression
壓縮後（80 tokens）：
"MLM 隨機遮住 15% 的詞彙，訓練模型根據上下文預測被遮住的詞。"
```

**優點：**
- 大幅減少 token 消耗（降低 API 費用）
- 去除雜訊 → LLM 回答更聚焦準確
- 相同 context window 可容納更多相關 chunk

**缺點：**
- 每個 chunk 需要額外一次 LLM 呼叫（成本 vs 節省需要評估）
- 壓縮品質依賴 LLM，可能誤刪相關內容

In [ ]:
# ── Part 3：上下文壓縮實作 ────────────────────────────

def contextual_compress(query: str, chunk: str, min_length: int = 30) -> Optional[str]:
    """
    使用 LLM 將 chunk 壓縮為只包含與 query 相關的部分。
    如果整段都不相關，返回 None。
    """
    prompt = f"""從以下文件中，提取與問題直接相關的句子或段落。
如果整段都不相關，只回答「無相關內容」四個字，不要有其他輸出。
不要新增任何文件中沒有的資訊，只能引用原文。

問題：{query}

文件：
{chunk}

只輸出相關的原文內容（或「無相關內容」）："""

    result = ask_llm(prompt, system="你是一個精準的文字抽取助理，任務是從文件中找出與問題相關的原文句子。")
    result = result.strip()

    # 如果回答是「無相關內容」或太短，視為無相關
    if "無相關內容" in result or len(result) < min_length:
        return None
    return result


def retrieve_and_compress(
    query: str,
    collection: object,
    n_retrieve: int = 5,
    final_k: int = 3
) -> Dict:
    """
    先向量搜索取回更多候選，再用 LLM 壓縮每個 chunk，
    過濾掉不相關的，最後返回 final_k 個壓縮後的 context。
    """
    # 向量搜索
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_retrieve,
        include=["documents", "distances"]
    )
    candidates = results["documents"][0]
    distances = results["distances"][0]

    # 壓縮每個候選
    compressed_results = []
    for i, (chunk, dist) in enumerate(zip(candidates, distances)):
        original_tokens = count_tokens(chunk)
        compressed = contextual_compress(query, chunk)

        if compressed:
            compressed_tokens = count_tokens(compressed)
            compressed_results.append({
                "original": chunk,
                "compressed": compressed,
                "original_tokens": original_tokens,
                "compressed_tokens": compressed_tokens,
                "similarity": round(1 - dist, 4),
                "compression_ratio": round(compressed_tokens / original_tokens, 3)
            })

        if len(compressed_results) >= final_k:
            break

    return {
        "query": query,
        "candidates_retrieved": len(candidates),
        "relevant_after_compression": len(compressed_results),
        "results": compressed_results
    }


# 示範
query = "KV Cache 在 LLM 推理中的作用是什麼？"
print(f"查詢問題：{query}")
print("=" * 70)
print("正在執行壓縮（需呼叫多次 LLM）...\n")

compress_output = retrieve_and_compress(
    query, std_collection, n_retrieve=5, final_k=3
)

print(f"取回候選 chunk 數：{compress_output['candidates_retrieved']}")
print(f"壓縮後相關 chunk 數：{compress_output['relevant_after_compression']}")
print()

total_orig_tokens = 0
total_comp_tokens = 0

for i, res in enumerate(compress_output['results']):
    total_orig_tokens += res['original_tokens']
    total_comp_tokens += res['compressed_tokens']
    print(f"【Chunk {i+1}】相似度：{res['similarity']:.4f}")
    print(f"  原始：{res['original_tokens']} tokens → 壓縮後：{res['compressed_tokens']} tokens（{res['compression_ratio']*100:.0f}%）")
    print(f"  壓縮後內容：{res['compressed'][:150]}")
    print()

print(f"{'─'*70}")
print(f"總 token 節省：{total_orig_tokens} → {total_comp_tokens} tokens")
if total_orig_tokens > 0:
    print(f"節省比例：{(1 - total_comp_tokens/total_orig_tokens)*100:.1f}%")

In [ ]:
# ── Part 3：Token 節省視覺化 ──────────────────────────

if compress_output['results']:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # 左圖：每個 chunk 的 token 比較
    results_list = compress_output['results']
    x = [f"Chunk {i+1}" for i in range(len(results_list))]
    orig_tokens = [r['original_tokens'] for r in results_list]
    comp_tokens = [r['compressed_tokens'] for r in results_list]

    x_pos = range(len(x))
    width = 0.35
    axes[0].bar([p - width/2 for p in x_pos], orig_tokens, width, label='原始 chunk', color='#FF6B6B', alpha=0.8)
    axes[0].bar([p + width/2 for p in x_pos], comp_tokens, width, label='壓縮後', color='#4ECDC4', alpha=0.8)
    axes[0].set_xticks(list(x_pos))
    axes[0].set_xticklabels(x)
    axes[0].set_ylabel('Token 數量')
    axes[0].set_title('Contextual Compression：Token 節省效果')
    axes[0].legend()
    for p, o, c in zip(x_pos, orig_tokens, comp_tokens):
        axes[0].text(p - width/2, o + 2, str(o), ha='center', va='bottom', fontsize=9)
        axes[0].text(p + width/2, c + 2, str(c), ha='center', va='bottom', fontsize=9)

    # 右圖：總體節省
    total_o = sum(orig_tokens)
    total_c = sum(comp_tokens)
    saved = total_o - total_c
    axes[1].pie(
        [total_c, saved],
        labels=[f'保留\n{total_c} tokens', f'節省\n{saved} tokens'],
        colors=['#4ECDC4', '#FFE66D'],
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': 10}
    )
    axes[1].set_title('整體 Token 節省比例')

    plt.tight_layout()
    plt.savefig('/tmp/nb06_compression.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("圖表已儲存")
else:
    print("（無足夠結果繪製圖表）")

---

## Part 4：Lost in the Middle 問題

### 核心概念

Liu et al.（2023）的研究「Lost in the Middle: How Language Models Use Long Contexts」
揭示了 LLM 處理長 context 時的一個重要偏差：

**當相關資訊位於 context 中間時，模型回答正確率最低。**

```
Context 位置效應（示意）：

正確率
  ↑
100% ┤████████                              ███████
 80% ┤         ████                    ████
 60% ┤              ████          ████
 40% ┤                   ████████            ← 中間區域準確率最低
  0% ┼────────────────────────────────────────→
      開頭                中間                結尾
```

**為什麼會這樣？** 推測與模型的訓練模式有關：
- 文章開頭通常是重要摘要（模型學會重視）
- 文章結尾通常是結論（模型學會重視）
- 中間內容往往是細節，注意力較弱

### 緩解策略

1. **將最相關的 chunk 放在 context 開頭**（按相似度降序排列）
2. **使用 Reranking**（NB03 介紹）確保最相關的在前
3. **減少 chunk 數量**（只傳前 2-3 個，避免中間區域存在）
4. **使用特殊 prompt**，明確告訴模型相關資訊的位置

In [ ]:
# ── Part 4：Lost in the Middle 示範 ──────────────────

# 準備材料：一個「針尖事實」和一些填充用的不相關段落
RELEVANT_FACT = (
    "Chinchilla 定律建議：模型參數量每增加一倍，訓練 token 數也應增加一倍。"
    "以 Llama 2 70B 為例，它使用了 2 兆個 token 進行訓練。"
)

FILLER_CHUNKS = [
    "Transformer 模型由 Vaswani 等人於 2017 年提出，核心機制是自注意力，能夠讓模型在處理每個詞時同時考慮序列中所有其他詞的資訊。",
    "量化技術將模型權重從 FP32 或 FP16 壓縮到 INT8 或 INT4，大幅降低記憶體需求和推理延遲。",
    "RoPE（旋轉位置嵌入）由 Su 等人提出，能夠更好地外推至訓練時未見過的更長序列。LLaMA、Mistral 等開源模型都採用 RoPE。",
    "指令微調使用大量指令-回應對進行監督式學習，讓模型學習如何按照人類指令回應。Stanford Alpaca 使用 GPT-4 生成 52,000 筆指令資料。",
    "KV Cache 快取已計算過的 Key 和 Value 矩陣，避免在生成每個新 token 時重複計算，是 LLM 推理加速的最基本也最重要的技術。",
]

QUESTION = "Chinchilla 定律對模型訓練有什麼建議？Llama 2 70B 用了多少 token 訓練？"

def build_context_with_position(relevant: str, fillers: List[str], position: str) -> str:
    """將相關段落放在 context 的不同位置（開頭、中間、結尾）"""
    if position == 'first':
        chunks = [relevant] + fillers
    elif position == 'middle':
        mid = len(fillers) // 2
        chunks = fillers[:mid] + [relevant] + fillers[mid:]
    else:  # last
        chunks = fillers + [relevant]

    parts = []
    for i, chunk in enumerate(chunks):
        parts.append(f"[參考資料 {i+1}]\n{chunk}")
    return "\n\n".join(parts)


def test_position_effect(question: str, relevant: str, fillers: List[str]) -> Dict:
    """測試相關資訊在不同位置時，LLM 回答品質"""
    results = {}

    for pos in ['first', 'middle', 'last']:
        context = build_context_with_position(relevant, fillers, pos)

        prompt = f"""根據以下參考資料回答問題。

{context}

問題：{question}
請根據參考資料回答，並引用具體數字或細節："""

        answer = ask_llm(prompt)
        results[pos] = {
            "answer": answer,
            "contains_2t": "2 兆" in answer or "2兆" in answer or "two trillion" in answer.lower(),
            "contains_chinchilla": "Chinchilla" in answer or "一倍" in answer,
        }
        print(f"位置：{pos:8s} | 包含 '2兆 tokens'：{'✅' if results[pos]['contains_2t'] else '❌'} | 包含 Chinchilla 細節：{'✅' if results[pos]['contains_chinchilla'] else '❌'}")

    return results


print(f"測試問題：{QUESTION}")
print(f"相關資訊位於：開頭 / 中間 / 結尾 三種情況")
print("=" * 70)
position_results = test_position_effect(QUESTION, RELEVANT_FACT, FILLER_CHUNKS)

In [ ]:
# ── Part 4：位置效應視覺化 ────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：Liu et al. 研究的示意圖（概念圖，非真實數據）
positions = ['1st', '2nd', '3rd', '4th (Mid)', '5th', '6th', '7th', '8th', '9th', '10th (Last)']
# 示意曲線：開頭和結尾準確率高，中間低（U 型曲線）
accuracy = [0.83, 0.72, 0.65, 0.54, 0.50, 0.51, 0.55, 0.68, 0.76, 0.81]

x = range(len(positions))
ax1.plot(x, accuracy, 'o-', color='#FF6B6B', linewidth=2, markersize=8)
ax1.fill_between(x, accuracy, min(accuracy), alpha=0.15, color='#FF6B6B')
ax1.axhline(y=min(accuracy), color='gray', linestyle='--', alpha=0.5, label=f'最低點 ({min(accuracy):.2f})')
ax1.axhline(y=max(accuracy), color='green', linestyle='--', alpha=0.5, label=f'最高點 ({max(accuracy):.2f})')
ax1.set_xticks(list(x))
ax1.set_xticklabels(positions, rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('回答正確率（示意）')
ax1.set_title('Lost in the Middle 效應\n（Liu et al., 2023 示意）')
ax1.set_ylim(0.3, 1.0)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.annotate('危險區域\n（相關資訊\n在此最易被忽略）',
             xy=(3.5, min(accuracy)), xytext=(5, 0.38),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=9, color='red', ha='center')

# 右圖：本次實驗結果
pos_labels = ['開頭\n(First)', '中間\n(Middle)', '結尾\n(Last)']
pos_keys = ['first', 'middle', 'last']
scores = []
for k in pos_keys:
    r = position_results.get(k, {})
    score = sum([r.get('contains_2t', False), r.get('contains_chinchilla', False)])
    scores.append(score)

colors = ['#4ECDC4' if s == max(scores) else '#FF6B6B' if s == min(scores) else '#FFE66D' for s in scores]
bars = ax2.bar(pos_labels, scores, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('答題得分（0-2 分）')
ax2.set_title('本次實驗：不同位置的回答品質\n（2 = 完整回答，0 = 遺漏關鍵資訊）')
ax2.set_ylim(0, 2.5)
for bar, score in zip(bars, scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{score}/2', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/nb06_lost_in_middle.png', dpi=100, bbox_inches='tight')
plt.show()
print("圖表已儲存")

### Part 4：緩解 Lost in the Middle 的實踐建議

```python
# ✅ 最佳實踐：按相似度降序排列 chunks，最相關的放最前面
def sort_chunks_by_relevance(retrieved_chunks):
    """確保最相關的 chunk 在 context 開頭"""
    return sorted(retrieved_chunks, key=lambda x: x['similarity'], reverse=True)

# ✅ 另一種方法：「三明治」排列 — 最相關的放首尾，次要的放中間
def sandwich_arrangement(chunks):
    """將最重要的 chunk 放在首尾（利用 U 型效應）"""
    if len(chunks) <= 2:
        return chunks
    # 最相關放第一個，第二相關放最後一個，其餘填中間
    return [chunks[0]] + chunks[2:] + [chunks[1]]
```

**關鍵面試觀點：**
> 「我們在生產 RAG 系統中，會確保根據 reranker 分數對 chunks 降序排列，
> 並將 top-1 chunk 永遠放在 context 開頭。這個簡單的調整讓我們的回答準確率提升了約 8%。」

---

## Part 5：Map-Reduce 與 Refine（超長文件問答）

### 核心概念

當文件長到無法放入 context window（例如 50 頁 PDF），
傳統 RAG 的「切塊 + 向量搜索」依然有效，但有些問題需要「跨全文推理」。
此時需要文件級別的問答策略：

**Map-Reduce 模式：**
```
長文件（50,000 tokens）
  ↓ 切成 N 個 chunks
[Chunk 1] → 提取相關資訊 → Summary 1      ← Map 階段（可並行）
[Chunk 2] → 提取相關資訊 → Summary 2
[Chunk N] → 提取相關資訊 → Summary N
                ↓
  Reduce：彙整所有 Summary → 最終答案
```

**Refine 模式：**
```
[Chunk 1] → 初步答案
         → [Chunk 2] → 精煉答案
                    → [Chunk 3] → 精煉答案
                               → ...       → 最終答案
```

| 面向 | Map-Reduce | Refine |
|------|-----------|--------|
| 並行化 | ✅ 可並行 Map 階段 | ❌ 必須串行 |
| 跨段落推理 | ❌ 每段獨立提取 | ✅ 答案逐漸累積 |
| 適合問題類型 | 事實聚合、統計 | 需要推理鏈的問題 |
| LLM 呼叫次數 | N（Map）+ 1（Reduce） | N（每 chunk 一次） |

In [ ]:
# ── Part 5：Map-Reduce 實作 ───────────────────────────

def map_extract(question: str, chunk: str) -> Optional[str]:
    """Map 階段：從單一 chunk 中提取與問題相關的資訊"""
    prompt = f"""請從以下文件片段中，提取與問題相關的資訊。
如果此片段不包含相關資訊，回答「此段無相關資訊」。
僅提取事實，不要生成答案。

問題：{question}

文件片段：
{chunk}

提取的相關資訊："""

    result = ask_llm(prompt)
    if "無相關資訊" in result or len(result.strip()) < 20:
        return None
    return result.strip()


def reduce_summaries(question: str, summaries: List[str]) -> str:
    """Reduce 階段：彙整所有 Map 階段的提取結果，生成最終答案"""
    combined = "\n\n".join([f"[來源 {i+1}] {s}" for i, s in enumerate(summaries)])

    prompt = f"""根據以下從文件各段落提取的資訊，
綜合回答問題。如有衝突資訊，請指出。

問題：{question}

各段落提取的資訊：
{combined}

綜合回答："""

    return ask_llm(prompt)


def map_reduce_qa(question: str, long_text: str, chunk_size: int = 600) -> Dict:
    """完整的 Map-Reduce 問答流程"""
    chunks = split_text_by_tokens(long_text, chunk_size=chunk_size, overlap=20)
    print(f"文件切成 {len(chunks)} 個 chunks（≈{chunk_size} tokens/塊）")
    print("Map 階段：提取各 chunk 的相關資訊...")

    # Map 階段
    summaries = []
    for i, chunk in enumerate(chunks):
        summary = map_extract(question, chunk)
        if summary:
            summaries.append(summary)
            print(f"  Chunk {i+1}/{len(chunks)}: ✅ 找到相關資訊（{count_tokens(summary)} tokens）")
        else:
            print(f"  Chunk {i+1}/{len(chunks)}: ── 無相關資訊")

    print(f"\nReduce 階段：彙整 {len(summaries)} 個相關段落的資訊...")

    # Reduce 階段
    if not summaries:
        final_answer = "文件中找不到相關資訊。"
    else:
        final_answer = reduce_summaries(question, summaries)

    return {
        "question": question,
        "total_chunks": len(chunks),
        "relevant_chunks": len(summaries),
        "summaries": summaries,
        "final_answer": final_answer,
        "total_input_tokens": count_tokens(long_text),
        "llm_calls": len(chunks) + (1 if summaries else 0)
    }


# 執行 Map-Reduce
mr_question = "這篇文章介紹了哪些 LLM 推理效率優化技術？每種技術的核心做法是什麼？"
print(f"問題：{mr_question}")
print("=" * 70)
mr_result = map_reduce_qa(mr_question, DEMO_ARTICLE, chunk_size=500)

print("\n" + "=" * 70)
print(f"最終答案：\n{mr_result['final_answer']}")
print(f"\n統計：使用 {mr_result['llm_calls']} 次 LLM 呼叫，\n"
      f"處理了 {mr_result['total_input_tokens']} tokens 的文件")

In [ ]:
# ── Part 5：Refine 實作 ──────────────────────────────

def refine_answer(question: str, current_answer: str, new_chunk: str) -> str:
    """Refine 步驟：根據新的 chunk 精煉現有答案"""
    prompt = f"""你有一個針對問題的現有答案。
現在有新的文件片段可以參考。
如果新資訊能改善或補充現有答案，請更新答案；否則維持原答案不變。

問題：{question}

現有答案：
{current_answer}

新的文件片段：
{new_chunk}

精煉後的答案："""

    return ask_llm(prompt)


def refine_qa(question: str, long_text: str, chunk_size: int = 600) -> Dict:
    """Refine 問答流程（串行，逐步精煉）"""
    chunks = split_text_by_tokens(long_text, chunk_size=chunk_size, overlap=20)
    print(f"文件切成 {len(chunks)} 個 chunks，開始逐步精煉...")

    current_answer = "目前沒有足夠資訊回答。"
    answer_history = []

    for i, chunk in enumerate(chunks):
        new_answer = refine_answer(question, current_answer, chunk)
        changed = new_answer.strip() != current_answer.strip()
        print(f"  Chunk {i+1}/{len(chunks)}: {'✏️  答案已更新' if changed else '── 答案不變'}")
        current_answer = new_answer
        answer_history.append(current_answer)

    return {
        "question": question,
        "total_chunks": len(chunks),
        "final_answer": current_answer,
        "answer_history": answer_history,
        "llm_calls": len(chunks)
    }


# 使用相同問題，Refine 模式
refine_question = "LLM 中的 KV Cache 是什麼？它解決了什麼問題？"
print(f"問題：{refine_question}")
print("=" * 70)
refine_result = refine_qa(refine_question, DEMO_ARTICLE, chunk_size=500)

print("\n" + "=" * 70)
print(f"最終答案：\n{refine_result['final_answer']}")
print(f"\n統計：使用 {refine_result['llm_calls']} 次 LLM 呼叫")

In [ ]:
# ── Part 5：Map-Reduce vs Refine 比較圖 ──────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左圖：Map-Reduce 流程示意
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Map-Reduce 模式', fontsize=12, fontweight='bold')

# 畫 chunks
chunk_colors = ['#AED6F1', '#A9DFBF', '#FAD7A0', '#F9E79F']
for i in range(4):
    y_pos = 8.5 - i * 1.8
    rect = plt.Rectangle((0.2, y_pos - 0.35), 2.5, 0.7,
                          facecolor=chunk_colors[i], edgecolor='gray', linewidth=1)
    ax.add_patch(rect)
    ax.text(1.45, y_pos, f'Chunk {i+1}', ha='center', va='center', fontsize=9)
    # 箭頭到 summary
    ax.annotate('', xy=(5.0, y_pos), xytext=(2.7, y_pos),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))
    # summary 框
    rect2 = plt.Rectangle((5.0, y_pos - 0.3), 2.2, 0.6,
                           facecolor='#D5DBDB', edgecolor='gray', linewidth=1)
    ax.add_patch(rect2)
    ax.text(6.1, y_pos, f'Summary {i+1}', ha='center', va='center', fontsize=8)

# 彙整箭頭
for i in range(4):
    y_pos = 8.5 - i * 1.8
    ax.annotate('', xy=(7.8, 5.0), xytext=(7.2, y_pos),
                arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1))

# Reduce 框
rect_r = plt.Rectangle((7.8, 4.4), 1.9, 1.2,
                        facecolor='#FADBD8', edgecolor='#E74C3C', linewidth=2)
ax.add_patch(rect_r)
ax.text(8.75, 5.0, 'Reduce\n(最終答案)', ha='center', va='center', fontsize=8)
ax.text(3.5, 0.3, '✅ 可並行  ❌ 損失跨段落關係', ha='center', fontsize=9, color='#555')

# 右圖：Refine 流程示意
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Refine 模式', fontsize=12, fontweight='bold')

answer_colors = ['#D5F5E3', '#A9DFBF', '#76D7C4', '#1ABC9C']
for i in range(4):
    y_top = 9.0 - i * 2.0
    # chunk 框
    rect_c = plt.Rectangle((0.2, y_top - 0.35), 2.0, 0.7,
                            facecolor='#AED6F1', edgecolor='gray', linewidth=1)
    ax2.add_patch(rect_c)
    ax2.text(1.2, y_top, f'Chunk {i+1}', ha='center', va='center', fontsize=9)

    # answer 框
    rect_a = plt.Rectangle((4.5, y_top - 0.35), 3.5, 0.7,
                            facecolor=answer_colors[i], edgecolor='gray', linewidth=1)
    ax2.add_patch(rect_a)
    label = '初始答案' if i == 0 else f'精煉答案 {i}'
    ax2.text(6.25, y_top, label, ha='center', va='center', fontsize=8)

    # chunk → answer 箭頭
    ax2.annotate('', xy=(4.5, y_top), xytext=(2.2, y_top),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

    # 上一個 answer → 下一個 answer 箭頭
    if i < 3:
        y_next = 9.0 - (i+1) * 2.0
        ax2.annotate('', xy=(6.25, y_next + 0.35), xytext=(6.25, y_top - 0.35),
                     arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5))

ax2.text(5.0, 0.3, '❌ 必須串行  ✅ 保留累積上下文', ha='center', fontsize=9, color='#555')

plt.tight_layout()
plt.savefig('/tmp/nb06_mapreduce_refine.png', dpi=100, bbox_inches='tight')
plt.show()
print("圖表已儲存")

---

## 技術選擇指南

| 技術 | 最佳適用場景 | 優點 | 缺點 |
|------|------------|------|------|
| **Parent-Child Chunking** | 技術文件、百科、FAQ，需要「精確匹配 + 豐富 context」 | 精準搜索 + 完整 context | 索引成本高，child+parent 兩層管理 |
| **Sentence-Window** | 答案是單一精確事實，需要前後句輔助理解 | 最精準的向量匹配 | 窗口可能截斷段落邊界 |
| **Contextual Compression** | 文件雜訊多，token 成本敏感 | 節省大量 token，減少 LLM 雜訊干擾 | 需要額外 LLM 呼叫，可能誤刪相關內容 |
| **Lost in the Middle 緩解** | 所有 RAG 系統都應套用 | 簡單（排序），顯著提升準確率 | 無明顯缺點（只是排序策略） |
| **Map-Reduce** | 超長文件，問題答案散落各處，需要聚合 | 可並行，適合統計/彙整型問題 | 損失跨段落推理關係 |
| **Refine** | 問題需要累積推理，答案需要逐步精煉 | 保留跨段落的推理鏈 | 速度慢（串行），成本高 |

### 決策樹

```
文件長度？
├── 超長（> context window）
│   ├── 問題需要跨全文推理？→ Refine
│   └── 問題可以分段回答？ → Map-Reduce
└── 一般長度（可放入 context window）
    ├── 答案是精確單句事實？ → Sentence-Window
    ├── 文件雜訊很多？        → Contextual Compression
    ├── 需要精準 + 豐富 context？ → Parent-Child
    └── 一般問答？            → 標準 RAG + 排序優化
```

---

## FDE 面試 Q&A

---

**Q1：Parent-child chunking 如何解決 chunk size 的兩難困境？**

> **A：** 傳統 RAG 面對兩難：小 chunk 向量搜索精準但 LLM context 不足，大 chunk context 豐富但向量搜索雜訊多。Parent-Child Chunking 通過「分離搜索單元與生成單元」解決這個問題：小的 child chunk（~100-150 tokens）只用於向量嵌入和搜索，搜索命中後自動取回對應的大 parent chunk（~500-600 tokens）送給 LLM。這樣同時享有精準的向量搜索和豐富的生成 context。額外代價是索引空間和建立時間增加，但在生產環境中通常值得。

---

**Q2：如果要處理一份 50 頁的 PDF，你的 RAG 方案是什麼？**

> **A：** 我會根據問題類型採用不同策略：
> 1. **大多數問題（「文件中提到了什麼？」）**：標準 RAG + Parent-Child Chunking 已足夠。50 頁 PDF 大約 25,000-40,000 tokens，我不需要全文放入 context，只需向量搜索後取回最相關的 2-3 個 parent chunks 即可。
> 2. **需要全文理解的問題（「請總結整份報告的核心論點」）**：使用 Map-Reduce — 每個 chunk 提取相關資訊，最後 reduce 成完整答案。
> 3. **需要累積推理的問題（「作者在第 3 章論點如何與第 7 章結論連結？」）**：使用 Refine 模式。
> 此外，我會在 PDF 解析時加入 metadata（頁碼、章節標題），讓過濾更精準。

---

**Q3：Context window 滿了怎麼辦？有哪些降低 token 使用量的策略？**

> **A：** 我會依序考慮以下策略：
> 1. **Contextual Compression**：用 LLM 壓縮每個 chunk，只保留與 query 相關的句子，通常能減少 60-80% 的 token 用量。
> 2. **減少 k 值**：從 5 個 chunks 降到 2-3 個，搭配 Reranker 確保保留的是最相關的。
> 3. **Parent-Child 取代 flat chunking**：child 搜索後只取一個 parent，而非多個大 chunk 全部送入。
> 4. **Map-Reduce**：如果問題本質上需要多段文字，改用 Map-Reduce 架構，每次只在一個 chunk 的 context 內操作。
> 5. **System prompt 精簡**：壓縮 system prompt 和 few-shot examples（如果有的話）。

---

**Q4：什麼是 Lost in the Middle 問題？你如何在生產系統中緩解它？**

> **A：** Liu et al.（2023）研究發現，LLM 在處理長 context 時，對位於中間的資訊注意力最弱，準確率呈 U 型曲線（開頭和結尾高，中間低）。這在 RAG 中意味著：如果最相關的 chunk 被排在中間，LLM 可能無法有效利用它。
> 緩解措施：
> 1. **排序策略**：用 reranker 分數對 chunks 降序排列，最相關的永遠放第一個。
> 2. **減少 chunks 數量**：只取 top-2 或 top-3（不是 top-10），讓「中間區域」越短越好。
> 3. **三明治排列**（進階）：最相關放開頭，次相關放結尾，其餘填中間，利用 U 型效應。
> 4. **Prompt 提示**：明確告訴模型「特別注意第 X 段資料」。

---

## 本 Notebook 總結

### 學到的五大技術

| # | 技術 | 解決的問題 | 核心機制 |
|---|------|-----------|----------|
| 1 | **Parent-Child Chunking** | chunk size 兩難 | 小塊搜索 + 自動取回大塊 |
| 2 | **Sentence-Window Retrieval** | 精確事實定位 | 句子嵌入 + 鄰近句子窗口 |
| 3 | **Contextual Compression** | Context 雜訊多、token 浪費 | LLM 提取相關原文 |
| 4 | **Lost in the Middle 緩解** | Context 位置偏差 | 按相似度排序，相關內容置前 |
| 5 | **Map-Reduce / Refine** | 超長文件無法放入 context | 分段提取 + 彙整 / 逐步精煉 |

### 關鍵設計原則

1. **分離搜索與生成**：向量搜索不等於 LLM 的最佳輸入，兩者可以分開優化
2. **Token 意識**：每個 token 都有成本，Compression 和排序都是 token 管理策略
3. **位置不是中性的**：Context 中的位置影響 LLM 的注意力分配
4. **沒有萬能方案**：根據文件長度、問題類型選擇合適的策略組合

### 下一步

- **NB07**：RAG 系統架構設計 — 如何將這些技術組合成生產級系統
- **NB08**：多文件 RAG — 處理異質文件來源（PDF + Web + DB）
- **回顧 NB03**：Reranking 技術是改善 Context 品質的重要前置步驟